<a href="https://colab.research.google.com/github/EUNTELLA/baseball/blob/main/test_0817_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

!python -c "import catboost; print('catboost:', catboost.__version__)"
!python -c "import torch; print('torch cuda:', torch.cuda.is_available()); print('cuda:', torch.version.cuda)"

Mon Aug 17 10:34:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q catboost==1.2.8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.7 MB/s eta 0:00:00


In [3]:
import catboost
from catboost import CatBoostClassifier

print("CatBoost:", catboost.__version__)

model = CatBoostClassifier(
    iterations=2,
    task_type="GPU",
    devices="0",
    verbose=False,
)
model.fit([[0], [1], [2], [3]], [0, 0, 1, 1])

print("CatBoost GPU 정상")

CatBoost: 1.2.8
CatBoost GPU 정상


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf /content/baseball /content/dataset /content/league_build
!git clone https://github.com/EUNTELLA/baseball.git /content/baseball
!unzip -q -o /content/drive/MyDrive/open.zip -d /content/dataset

Cloning into '/content/baseball'...
remote: Enumerating objects: 596, done.
remote: Counting objects: 100% (596/596), done.
remote: Compressing objects: 100% (393/393), done.


In [6]:
%cd /content/baseball
!git pull

!python /content/baseball/0817/01_catboost_multiseason_tuning_colab.py \
  --train /content/dataset/data/train.csv \
  --output /content/drive/MyDrive/0817_catboost_multiseason_tuning.json \
  --task-type GPU

/content/baseball
Already up to date.
--- 1단계: 7개 설정 × 3개 시즌 × 1시드 ---
d6_lr05_l2_1_baseline fold=2022 iter=314 raw=2344.43 centered=2351.38 sec=31.1
d6_lr05_l2_1_baseline fold=2023 iter=0 raw=1.83 centered=2.50 sec=12.8
d6_lr05_l2_1_baseline fold=2024 iter=285 raw=755.06 centered=786.19 sec=51.4
d5_lr05_l2_1 fold=2022 iter=326 raw=2340.88 centered=2346.37 sec=26.3
d5_lr05_l2_1 fold=2023 iter=1 raw=10.22 centered=12.52 sec=11.7
d5_lr05_l2_1 fold=2024 iter=297 raw=754.41 centered=787.32 sec=42.6
d7_lr05_l2_1 fold=2022 iter=323 raw=2355.93 centered=2360.35 sec=36.6
d7_lr05_l2_1 fold=2023 iter=0 raw=8.99 centered=9.77 sec=14.7
d7_lr05_l2_1 fold=2024 iter=199 raw=764.97 centered=789.44 sec=47.8
d6_lr05_l2_3 fold=2022 iter=207 raw=2339.96 centered=2345.51 sec=23.0
d6_lr05_l2_3 fold=2023 iter=0 raw=1.84 centered=2.51 sec=13.0
d6_lr05_l2_3 fold=2024 iter=239 raw=758.61 centered=788.09 sec=44.7
d6_lr05_l2_5 fold=2022 iter=323 raw=2350.83 centered=2357.51 sec=31.9
d6_lr05_l2_5 fold=2023 iter=0 

In [7]:
%cd /content/baseball
!git pull

!python /content/baseball/0817/02_catboost_tuned_build_colab.py \
  --train /content/dataset/data/train.csv \
  --test /content/dataset/data/test.csv \
  --sample /content/dataset/data/sample_submission.csv \
  --output-dir /content/drive/MyDrive/0817 \
  --task-type GPU

/content/baseball
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 3), reused 9 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 12.40 KiB | 488.00 KiB/s, done.
From https://github.com/EUNTELLA/baseball
   e54710d..ed52df2  main       -> origin/main
Updating e54710d..ed52df2
Fast-forward
 0817/02_catboost_tuned_build_colab.py              |  286 +++
 0817/README.md                                     |   17 +
 0817/results/0817_catboost_multiseason_tuning.json | 1987 ++++++++++++++++++++
 doc/EXPERIMENTS.md                                 |   11 +-
 4 files changed, 2300 insertions(+), 1 deletion(-)
 create mode 100644 0817/02_catboost_tuned_build_colab.py
 create mode 100644 0817/results/0817_catboost_multiseason_tuning.json
0:	learn: 0.6926105	test: 0.6930309	best: 0.6930309 (0)	total: 190ms	remaining: 6m 20s
100:	learn: 0.6824991	test: 0.6895263	best: 0.6895263 (

In [8]:
%cd /content/baseball
!git pull

!python /content/baseball/0817/03_catboost_full_pipeline_walkforward_colab.py \
  --train /content/dataset/data/train.csv \
  --output /content/drive/MyDrive/0817_full_pipeline_walkforward.json \
  --task-type GPU

/content/baseball
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 23 (delta 12), reused 19 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 3.19 MiB | 4.67 MiB/s, done.
From https://github.com/EUNTELLA/baseball
   ed52df2..514b3da  main       -> origin/main
Updating ed52df2..514b3da
Fast-forward
 0817/02_catboost_tuned_build_colab.py              |   4 +
 .../03_catboost_full_pipeline_walkforward_colab.py | 369 +++++++++++++++++++++
 0817/README.md                                     |  17 +
 .../submit_catboost_lr03_l2_3_train_only.json      |  92 +++++
 .../submit_catboost_lr03_l2_3_train_only.zip       | Bin 0 -> 3343276 bytes
 doc/EXPERIMENTS.md                                 |  25 ++
 6 files changed, 507 insertions(+)
 create mode 100644 0817/03_catboost_full_pipeline_walkforward_colab.py
 create mode 100644 0817/results/submit_catboost_lr03_l2_3_train_only.json


In [ ]:
%cd /content/baseball
!git pull

!python /content/baseball/0817/04_catboost_brier_regression_screen_colab.py \
  --train /content/dataset/data/train.csv \
  --output /content/drive/MyDrive/0817_catboost_brier_regression.json \
  --task-type GPU